In [0]:
from pyspark.sql import functions as F

spark.sql("USE plworkforce_catalog.003_gold")

# Load fact tables
fact_financials = spark.table("plworkforce_catalog.003_gold.fact_financials")
fact_payroll = spark.table("plworkforce_catalog.003_gold.fact_payroll")

# Load dimension tables
dim_account = spark.table("plworkforce_catalog.003_gold.dim_account")
dim_department = spark.table("plworkforce_catalog.003_gold.dim_department")
dim_company = spark.table("plworkforce_catalog.003_gold.dim_company")
dim_employee = spark.table("plworkforce_catalog.003_gold.dim_employee")

In [0]:
# Financial cube with all P&L metrics by company, department, account type, and time
financial_base = (
    fact_financials
    .join(dim_company, "company_id")
    .join(dim_department, "department_id")
    .join(dim_account, "account_id")
    .select(
        "company_id",
        "company_name",
        "industry",
        "country",
        "department_id",
        "department_name",
        fact_financials["account_type"],
        "category",
        "year",
        "month",
        "year_month",
        "net_amount"
    )
)

financial_cube = (
    financial_base
    .groupBy(
        "company_id",
        "company_name",
        "industry",
        "country",
        "department_id",
        "department_name",
        "account_type",
        "category",
        "year",
        "month",
        "year_month"
    )
    .agg(
        F.sum("net_amount").alias("total_amount"),
        F.count("*").alias("transaction_count"),
        F.avg("net_amount").alias("avg_transaction_amount")
    )
    .withColumn(
        "revenue",
        F.when(F.col("account_type") == "Revenue", F.col("total_amount")).otherwise(0)
    )
    .withColumn(
        "cogs",
        F.when(F.col("account_type") == "Cogs", F.col("total_amount")).otherwise(0)
    )
    .withColumn(
        "operating_expenses",
        F.when(F.col("account_type") == "Expense", F.col("total_amount")).otherwise(0)
    )
    .orderBy("company_name", "year_month", "department_name", "account_type")
)

display(financial_cube)

In [0]:
# HR cube with compensation and headcount metrics by company, department, position, and time
hr_base = (
    fact_payroll
    .join(dim_company, "company_id")
    .join(dim_department, "department_id")
    .select(
        "company_id",
        "company_name",
        "industry",
        "country",
        "department_id",
        "department_name",
        "position",
        "employee_status",
        "year",
        "month",
        "year_month",
        "employee_id",
        "gross_salary",
        "bonus",
        "overtime_pay",
        "commission",
        "allowances",
        "total_compensation"
    )
)

hr_cube = (
    hr_base
    .groupBy(
        "company_id",
        "company_name",
        "industry",
        "country",
        "department_id",
        "department_name",
        "position",
        "employee_status",
        "year",
        "month",
        "year_month"
    )
    .agg(
        F.countDistinct("employee_id").alias("headcount"),
        F.count("*").alias("payroll_records"),
        F.sum("total_compensation").alias("total_compensation_sum"),
        F.avg("total_compensation").alias("avg_compensation"),
        F.min("total_compensation").alias("min_compensation"),
        F.max("total_compensation").alias("max_compensation"),
        F.sum("gross_salary").alias("total_base_salary"),
        F.sum("bonus").alias("total_bonus"),
        F.sum("overtime_pay").alias("total_overtime"),
        F.sum("commission").alias("total_commission"),
        F.sum("allowances").alias("total_allowances"),
        F.avg("gross_salary").alias("avg_base_salary"),
        F.avg("bonus").alias("avg_bonus"),
        F.avg("overtime_pay").alias("avg_overtime"),
        F.avg("commission").alias("avg_commission")
    )
    .withColumn(
        "overtime_to_base_ratio_pct",
        (F.col("total_overtime") / F.col("total_base_salary")) * 100
    )
    .withColumn(
        "bonus_to_base_ratio_pct",
        (F.col("total_bonus") / F.col("total_base_salary")) * 100
    )
    .withColumn(
        "variable_pay_pct",
        ((F.col("total_bonus") + F.col("total_overtime") + F.col("total_commission")) / 
         F.col("total_compensation_sum")) * 100
    )
    .orderBy("company_name", "year_month", "department_name", "position")
)

display(hr_cube)

In [0]:
# Save financial cube as permanent table
financial_cube.write.format("delta").mode("overwrite").saveAsTable("plworkforce_catalog.003_gold.cube_financial")

# Save HR cube as permanent table
hr_cube.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("plworkforce_catalog.003_gold.cube_hr_workforce")

print("Cubes saved to:")
print("  - plworkforce_catalog.003_gold.cube_financial")
print("  - plworkforce_catalog.003_gold.cube_hr_workforce")